In [ ]:
# SECTION 
# testing bag of words cleaning and characterization. Consider: bigrams, lowercase, lemitization (room/rooms), sparse terms

# start from pre-filtered english reviews
# Clean everything and split to training and test set in later steps. 

import pandas as pd

df = pd.read_csv("../data/data_eng.csv", index_col=0)
print(df.head())

     Unnamed: 0 Review_Language City Name    Hotel Name  \
985           0         English    Ankara  Hotel Ickale   
986           1         English    Ankara  Hotel Ickale   
987           2         English    Ankara  Hotel Ickale   
988           3         English    Ankara  Hotel Ickale   
989           4         English    Ankara  Hotel Ickale   

                                         Upside_Review  \
985  Staff. Serkan türk was a great help during our...   
986  Nicely furnished room and nicely decorated lob...   
987  Very central, near 2 metro stations. Room was ...   
988  Lovely interior.. great location.. friendly staff   
989      Thanks to Mr.Serkan, our problems were solved   

                                       Downside_Review  Review_Score  \
985                                                NaN          10.0   
986                                                NaN           9.0   
987  The wifi had annoying login that required logg...           9.0   
988  wha

In [42]:
# Confirm the provided NA flags match the real missing values
print(df[["na_up_review", "na_down_review"]].dtypes)
print((df["na_up_review"] == df["Upside_Review"].isna()).all())
print((df["na_down_review"] == df["Downside_Review"].isna()).all())


na_up_review      bool
na_down_review    bool
dtype: object
True
False


In [43]:
# Noted there are some errors in downside reviews
up_mismatch = df[df["na_up_review"] != df["Upside_Review"].isna()]
down_mismatch = df[df["na_down_review"] != df["Downside_Review"].isna()]

print(len(up_mismatch), len(down_mismatch))
print(up_mismatch[["Upside_Review", "na_up_review", "source"]].head(20))
print(down_mismatch[["Downside_Review", "na_down_review", "source"]].head(20))

0 5
Empty DataFrame
Columns: [Upside_Review, na_up_review, source]
Index: []
      Downside_Review  na_down_review  source
11016             NaN           False       1
11116             NaN           False       1
11118             NaN           False       1
11419             NaN           False       1
11838             NaN           False       1


In [44]:
up_empty = df["Upside_Review"].fillna("").str.strip() == ""
down_empty = df["Downside_Review"].fillna("").str.strip() == ""

print((df["na_up_review"] == up_empty).all())
print((df["na_down_review"] == down_empty).all())

True
False


In [45]:
still_off = df[df["na_down_review"] != down_empty]
print(len(still_off))
print(still_off[["Downside_Review", "na_down_review", "source"]].apply(
    lambda col: col.map(repr) if col.name == "Downside_Review" else col))

5
      Downside_Review  na_down_review  source
11016             nan           False       1
11116             nan           False       1
11118             nan           False       1
11419             nan           False       1
11838             nan           False       1


In [46]:
# compute own flags
df["up_empty"] = df["Upside_Review"].fillna("").str.strip() == ""
df["down_empty"] = df["Downside_Review"].fillna("").str.strip() == ""

In [ ]:
# NOTE: using recomputed flags: provided na_* flags disagreed with the text in 5 AI rows

# Fill, combine, clean
# Normalize : replace nulls with empty strings so pandas can join, clean, and vectorize
df[["Upside_Review", "Downside_Review"]] = df[["Upside_Review", "Downside_Review"]].fillna("")

# Build clean text column: lowercase, clean up whitespace
df["text"] = (df["Upside_Review"] + " " + df["Downside_Review"]) \
    .str.lower() \
    .str.replace(r"\s+", " ", regex=True) \
    .str.strip()

print(df.head(3))

     Unnamed: 0 Review_Language City Name    Hotel Name  \
985           0         English    Ankara  Hotel Ickale   
986           1         English    Ankara  Hotel Ickale   
987           2         English    Ankara  Hotel Ickale   

                                         Upside_Review  \
985  Staff. Serkan türk was a great help during our...   
986  Nicely furnished room and nicely decorated lob...   
987  Very central, near 2 metro stations. Room was ...   

                                       Downside_Review  Review_Score  \
985                                                             10.0   
986                                                              9.0   
987  The wifi had annoying login that required logg...           9.0   

    Sentiment  source Prompt_Language  na_up_review  na_down_review  up_empty  \
985       POS       0             NaN         False            True     False   
986       POS       0             NaN         False            True     False  

In [ ]:

# Characterization checks. NOTE: using self computed fields after noting mismatch
# 0 = human, 1 = AI
print(df["source"].value_counts())                                  
print(df.groupby("source")[["up_empty", "down_empty"]].mean())
print(pd.crosstab(df["source"], df["Sentiment"]))

source
0    1000
1    1000
Name: count, dtype: int64
        up_empty  down_empty
source                      
0          0.067       0.174
1          0.010       0.052
Sentiment  NEG  POS
source             
0          500  500
1          500  500


In [ ]:
# make bag of words
# does count vectorizer ignore punctuation?
# make vocabulary

In [56]:
# FIXME split between training and test set, WILL NEED TO USE DATASET FROM BOBBY OR GG
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df, test_size=0.25, stratify=df["source"], random_state=42
)

In [58]:
from sklearn.feature_extraction.text import CountVectorizer

# fit_transform on the training text does two things: it learns the vocabulary (every distinct word that passes your filters)
#   , then counts each word in each review. 
#   transform on the test text only counts; it uses the training vocabulary, and words never seen in training are ignored. 
#   The results are sparse matrices with one row per review and one column per vocabulary word.
#   NOTE: naming convention is capital letters matrices and lowercase vector
#   min_df is important, it controls sparse term removal. A value of 5 means that only words appearing in at least 5 reviews are kept.

#   POSSIBLE EDITS 
#   keep stop words (e.g., the, was, very) as they might be indicators of AI writing?
#   to experiment with bigrams add something like ngram_range=(1, 2)
#   referencing hotel and city names could be AI indicator but adds complication 

# NOTE: tokenization default ignores punctuation and drops single character tokens. 
vectorizer = CountVectorizer(min_df=5)
X_train = vectorizer.fit_transform(train_df["text"])
X_test = vectorizer.transform(test_df["text"])
y_train, y_test = train_df["source"], test_df["source"]

In [ ]:
vocab = vectorizer.get_feature_names_out()

# this is the number of words in our vocabulary
print(len(vocab))
print(vocab[:50])

# Most frequent words per class
import numpy as np
counts = pd.DataFrame({
    # the np.asarray is housekeeping since we stored as a sparse matrix
    "human": np.asarray(X_train[y_train.values == 0].sum(axis=0)).ravel(),
    "ai":    np.asarray(X_train[y_train.values == 1].sum(axis=0)).ravel(),
}, index=vocab)
print(counts.sort_values("ai", ascending=False).head(5))
print(counts.sort_values("human", ascending=False).head(5))

1373
['10' '15' '20' '24' '30' '40' '50' 'able' 'about' 'above' 'absolute'
 'absolutely' 'ac' 'acceptable' 'access' 'accessible' 'accommodate'
 'accommodating' 'accommodation' 'accommodations' 'across' 'actual'
 'actually' 'add' 'added' 'addition' 'additional' 'additionally' 'address'
 'addressed' 'adds' 'adequate' 'adjacent' 'adults' 'advance' 'advertised'
 'advice' 'affect' 'after' 'afternoon' 'again' 'ages' 'air' 'aircon'
 'airport' 'all' 'allowing' 'almost' 'alpi' 'already']
     human    ai
the   2391  3968
and   1262  1883
was    993  1635
to     880  1015
of     456   749
     human    ai
the   2391  3968
and   1262  1883
was    993  1635
to     880  1015
in     583   657


In [ ]:
# NOTE: there are a few numerical responses. It is a choice whether to keep them.
#   It appears they are stronlgy included in human reviews, so they may indicate a non-AI review (i.e., an AI may not complain about a hidden 5 euro charge)

nums = [w for w in vocab if w.isdigit()]
print(counts.loc[nums])

    human  ai
10     25   0
15      7   0
20      7   0
24      5   2
30     10   0
40      5   0
50      6   0
